# Data Leakage Check — Retail Revenue Forecasting

**Mục đích:** Kiểm tra rò rỉ dữ liệu (data leakage) theo thời gian giữa các tập train/validation/test trước khi đưa vào modeling.

**Phạm vi kiểm tra:**
- Khoảng giao thời gian giữa train (2019–2022 / 2019–2023), val (2023 / 2024), test (2025)
- Tính liên tục và toàn vẹn của chuỗi thời gian (gap, duplicate, sort)
- Missing values ở target (`revenue`)
- Phân bố các đặc trưng liên quan giai đoạn COVID

**Note:** chưa check feature engineering leakage.

In [14]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / "data").exists():
    raise FileNotFoundError("Could not locate a 'data' directory above the current path.")

print(ROOT)


d:\project_git\retail-revenue-forecasting


In [15]:
DATE_COLS = ["week_start"]

files = {
    "train_1": "train_2019_2022.csv",
    "val_1":   "validation_2023.csv",
    "train_2": "train_2019_2023.csv",
    "val_2":   "validation_2024.csv",
    "test":    "test_2025.csv",
    "df":      "cleaned_transactions.csv",
}

data = {
    name: pd.read_csv(ROOT / "data/processed" / fname, parse_dates=DATE_COLS)
    for name, fname in files.items()
}

train_1, val_1, train_2, val_2, test, df = (
    data["train_1"], data["val_1"], data["train_2"],
    data["val_2"], data["test"], data["df"],
)

In [16]:
df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
df["split_year_original"] = df["split_year"]
df["split_year"] = df["week_start"].dt.year
train = df[df["split_year"] <= 2022]
val_2023 = df[df["split_year"] == 2023]
val_2024 = df[df["split_year"] == 2024]
test = df[df["split_year"] == 2025]

print("Train:", len(train), "weeks")
print("Validation 2023:", len(val_2023), "weeks")
print("Validation 2024:", len(val_2024), "weeks")
print("Test 2025:", len(test), "weeks")

print("\nDate ranges:")

for name, data in [
    ("Train", train),
    ("Validation 2023", val_2023),
    ("Validation 2024", val_2024),
    ("Test", test),
]:
    if len(data) > 0:
        print(
            f"{name:18s}: "
            f"{data['week_start'].min().date()} "
            f"→ "
            f"{data['week_start'].max().date()}"
        )

Train: 6816 weeks
Validation 2023: 1756 weeks
Validation 2024: 1728 weeks
Test 2025: 1700 weeks

Date ranges:
Train             : 2018-12-31 → 2022-12-26
Validation 2023   : 2023-01-02 → 2023-12-25
Validation 2024   : 2024-01-01 → 2024-12-30
Test              : 2025-01-06 → 2025-12-29


In [17]:
#train(29-22) ∩ val 23
overlap_train_val23 = set(train_1["week_start"]) & set(val_1["week_start"])
#train(29-23) ∩ val 24
overlap_train_val24 = set(train_2["week_start"]) & set(val_2["week_start"])
#val 24 ∩ test 25

overlap_val_test    = set(val_2["week_start"]) & set(test["week_start"])

assert not overlap_train_val23, f"Overlap: {overlap_train_val23}"
assert not overlap_train_val24, f"Overlap: {overlap_train_val24}"
assert not overlap_val_test, f"Overlap: {overlap_val_test}"

In [18]:

print("Missing week_start:", df["week_start"].isna().sum())
print("Min date:", df["week_start"].min())
print("Max date:", df["week_start"].max())

Missing week_start: 0
Min date: 2018-12-31 00:00:00
Max date: 2025-12-29 00:00:00


In [19]:
duplicate_count = df["week_start"].duplicated().sum()

print("Duplicate weeks:", duplicate_count)

Duplicate weeks: 11634


In [20]:
print("Duplicate full rows:", df.duplicated().sum())

Duplicate full rows: 0


In [21]:
is_sorted = df["week_start"].is_monotonic_increasing

print("Chronological:", is_sorted)


Chronological: True


In [22]:
week_diff = df["week_start"].diff().dt.days.dropna()

abnormal_gaps = week_diff[week_diff != 7]

print("Total week gaps checked:", len(week_diff))
print("Abnormal gaps:", len(abnormal_gaps))

if len(abnormal_gaps) == 0:
    print("All weeks are continuous")
else:
    print("Abnormal week gaps detected")

    print("\nExamples:")
    print(
        df.loc[
            abnormal_gaps.index,
            ["week_start"]
        ].head(10)
    )

Total week gaps checked: 11999
Abnormal gaps: 11634
Abnormal week gaps detected

Examples:
   week_start
1  2018-12-31
2  2018-12-31
3  2018-12-31
4  2018-12-31
5  2018-12-31
6  2018-12-31
7  2018-12-31
8  2018-12-31
9  2018-12-31
10 2018-12-31


In [23]:
print("Missing revenue:", df["revenue"].isna().sum())

Missing revenue: 0


In [24]:
covid_columns = [
        "is_covid_period",
        "is_covid_shock",
        "covid_phase"
    ]

for col in covid_columns:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False))


is_covid_period
is_covid_period
False    8831
True     3169
Name: count, dtype: int64

is_covid_shock
is_covid_shock
False    11390
True       610
Name: count, dtype: int64

covid_phase
covid_phase
post_covid          6847
covid_adaptation    2559
pre_covid           1984
covid_shock          610
Name: count, dtype: int64


## Kết luận

- Không phát hiện leakage về thời gian: $Train \cap Val = \emptyset$, $Train \cap Test = \emptyset$, $Val \cap Test = \emptyset$
- Dữ liệu liên tục, không có gap bất thường, không trùng lặp `week_start`
- Chuỗi thời gian đã được sắp xếp đúng thứ tự
- Không có missing value ở target (`revenue`)
